# 06 -- Hyperparameter Tuning

## Objective

Optimise the HistGradientBoosting classifier using cross-validation to improve predictive performance. Hyperparameter selection will use training data only, while the held-out test set will be reserved for final evaluation.

Import libraries and load the data

In [1]:
import time
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)

from pathlib import Path

pd.set_option("display.max_columns", None)

df = pd.read_csv(
    "../data/processed/loan_data_feature_engineered.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (44988, 14)


# Prepare the features and split the data

In [2]:
target = "Loan Status"

numeric_features = [
    "Age",
    "Employee Experience",
    "Log_Person_Income",
    "Loan Amount",
    "Credit History",
    "Credit Score",
    "Loan_to_Income"
]

categorical_features = [
    "Gender",
    "Education",
    "Home Ownership",
    "Loan Intent",
    "Previous Loan"
]

features = numeric_features + categorical_features

X = df[features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (35990, 12)
Test set: (8998, 12)


Build the preprocessing pipeline

In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

Define the search space

In [4]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            HistGradientBoostingClassifier(
                random_state=42,
                early_stopping=False
            )
        )
    ]
)

param_distributions = {
    "classifier__learning_rate": [
        0.03, 0.05, 0.08, 0.10, 0.15
    ],
    "classifier__max_iter": [
        100, 150, 200, 300
    ],
    "classifier__max_leaf_nodes": [
        7, 15, 23, 31
    ],
    "classifier__min_samples_leaf": [
        10, 20, 30, 50
    ],
    "classifier__l2_regularization": [
        0.0, 0.1, 1.0, 5.0
    ]
}

print("Hyperparameter combinations available:",
      np.prod([len(v) for v in param_distributions.values()]))

Hyperparameter combinations available: 1280


# Run cross-validation

In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=25,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True,
    return_train_score=True
)

start_time = time.time()

search.fit(X_train, y_train)

search_time = time.time() - start_time

print(f"\nSearch completed in {search_time:.2f} seconds.")
print("Best cross-validation ROC-AUC:",
      round(search.best_score_, 4))
print("\nBest parameters:")

for parameter, value in search.best_params_.items():
    print(f"{parameter}: {value}")

Fitting 5 folds for each of 25 candidates, totalling 125 fits

Search completed in 119.11 seconds.
Best cross-validation ROC-AUC: 0.9618

Best parameters:
classifier__min_samples_leaf: 30
classifier__max_leaf_nodes: 15
classifier__max_iter: 300
classifier__learning_rate: 0.15
classifier__l2_regularization: 0.1


# Evaluate the tuned model on the test set

In [6]:
best_model = search.best_estimator_

y_pred_tuned = best_model.predict(X_test)
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

tuned_results = {
    "Accuracy": accuracy_score(y_test, y_pred_tuned),
    "Precision": precision_score(
        y_test, y_pred_tuned, zero_division=0
    ),
    "Recall": recall_score(
        y_test, y_pred_tuned, zero_division=0
    ),
    "F1 Score": f1_score(
        y_test, y_pred_tuned, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(y_test, y_prob_tuned),
    "Average Precision": average_precision_score(
        y_test, y_prob_tuned
    )
}

tuned_results_df = pd.DataFrame(
    tuned_results.items(),
    columns=["Metric", "Tuned HistGradientBoosting"]
)

tuned_results_df["Tuned HistGradientBoosting"] = (
    tuned_results_df["Tuned HistGradientBoosting"].round(4)
)

display(tuned_results_df)

print("\nClassification report:")
print(classification_report(
    y_test, y_pred_tuned, zero_division=0
))

,Metric,Tuned HistGradientBoosting
0,Accuracy,0.9031
1,Precision,0.8294
2,Recall,0.7100
3,F1 Score,0.7651
4,ROC-AUC,0.9609
5,Average Precision,0.8932



Classification report:
              precision    recall  f1-score   support

           0       0.92      0.96      0.94      6998
           1       0.83      0.71      0.77      2000

    accuracy                           0.90      8998
   macro avg       0.87      0.83      0.85      8998
weighted avg       0.90      0.90      0.90      8998



# Compare with the original model

In [7]:
original_results = {
    "Accuracy": 0.9025,
    "Precision": 0.8529,
    "Recall": 0.6785,
    "F1 Score": 0.7558,
    "ROC-AUC": 0.9573,
    "Average Precision": 0.8856
}

comparison = pd.DataFrame({
    "Metric": list(original_results.keys()),
    "Original": list(original_results.values()),
    "Tuned": [
        tuned_results[m] for m in original_results.keys()
    ]
})

comparison["Change"] = (
    comparison["Tuned"] - comparison["Original"]
).round(4)

display(comparison.round(4))

,Metric,Original,Tuned,Change
0,Accuracy,0.9025,0.9031,0.0006
1,Precision,0.8529,0.8294,-0.0235
2,Recall,0.6785,0.7100,0.0315
3,F1 Score,0.7558,0.7651,0.0093
4,ROC-AUC,0.9573,0.9609,0.0036
5,Average Precision,0.8856,0.8932,0.0076


# Saving the tuned model and results

In [8]:
import joblib

output_dir = Path("../models")
reports_dir = Path("../reports")

output_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    best_model,
    output_dir / "hist_gradient_boosting_tuned.joblib"
)

tuned_results_df.to_csv(
    reports_dir / "tuned_hist_gradient_boosting_results.csv",
    index=False
)

comparison.to_csv(
    reports_dir / "original_vs_tuned_comparison.csv",
    index=False
)

print("Tuned model and results saved.")

Tuned model and results saved.


### Hyperparameter Tuning Results

HistGradientBoosting was tuned using randomised hyperparameter search with five-fold stratified cross-validation. The search evaluated 25 hyperparameter combinations and selected a configuration with a learning rate of 0.15, 300 boosting iterations, 15 maximum leaf nodes, a minimum of 30 observations per leaf, and an L2 regularisation value of 0.1.

The tuned model achieved a mean cross-validation ROC-AUC of 0.9618 and a test ROC-AUC of 0.9609. Test accuracy increased from 90.25% to 90.31%, while recall increased from 67.85% to 71.00% and the F1-score increased from 0.7558 to 0.7651. Average precision also improved from 0.8856 to 0.8932. Precision decreased from 85.29% to 82.94%, indicating a trade-off between identifying more class 1 observations and maintaining the reliability of class 1 predictions.

Overall, hyperparameter tuning produced a modest improvement in discrimination and a more balanced classification result. The tuned model will be retained as the leading candidate for subsequent probability calibration and decision-threshold evaluation.